# NpuKit — MNIST tiny-ViT (PYNQ-Z2)

Geometry: resize **28→16**, patch **4**, pair-average → **T=8**, **D=8**.

Train on the Docker host (torch):
```bash
python3 host/train_vit_mnist.py
```
Then copy `vit_mnist_weights.npz`, `mnist_sample.npz`, and this notebook to the board.

Float test accuracy is ~80%+; the int8/Q12 FPGA path is lower until better QAT — this notebook checks **ref vs board** match and batch accuracy.

In [1]:
import importlib
import sys

BIT = "/home/xilinx/jupyter_notebooks/npukit.bit"
sys.path.insert(0, "/home/xilinx/jupyter_notebooks")

import npukit_vit_mnist as vit

importlib.reload(vit)
print("T", vit.VIT_T, "D", vit.VIT_D)
print("weights", vit.DEFAULT_WEIGHTS, "exists", vit.DEFAULT_WEIGHTS.exists())
print("sample", vit.DEFAULT_SAMPLE, "exists", vit.DEFAULT_SAMPLE.exists())

T 8 D 8
weights /home/xilinx/jupyter_notebooks/vit_mnist_weights.npz exists True
sample /home/xilinx/jupyter_notebooks/mnist_sample.npz exists True


## Offline ref (trained weights + real MNIST sample)

In [2]:
rc = vit.run_vit_smoke(bit_path=None, seed=0, n=16)
assert rc == 0
print("ref-only return", rc)

loaded MNIST sample from /home/xilinx/jupyter_notebooks/mnist_sample.npz (n=16)
=== MNIST tiny-ViT smoke ===
IMG=16 PATCH=4 T=8 D=8 classes=10
scales ACT/W/P=64.0/64.0/127.0
weights=/home/xilinx/jupyter_notebooks/vit_mnist_weights.npz

--- image[0] label=6 ---
--- ref ---
ref pred=6 logits_q12[:4]=[12472, -7875, 12638, -3196]

--- image[1] label=8 ---
--- ref ---
ref pred=8 logits_q12[:4]=[8307, -5567, 5386, 1893]

--- image[2] label=3 ---
--- ref ---
ref pred=0 logits_q12[:4]=[10888, -13018, 7459, 6739]

--- image[3] label=0 ---
--- ref ---
ref pred=0 logits_q12[:4]=[17211, -14180, 3199, 868]

--- image[4] label=2 ---
--- ref ---
ref pred=2 logits_q12[:4]=[1061, 1848, 17056, 4143]

--- image[5] label=9 ---
--- ref ---
ref pred=0 logits_q12[:4]=[7639, -11523, -535, 3935]

--- image[6] label=0 ---
--- ref ---
ref pred=0 logits_q12[:4]=[17316, -16351, -1473, 4121]

--- image[7] label=2 ---
--- ref ---
ref pred=2 logits_q12[:4]=[6817, -6763, 17020, 4659]

--- image[8] label=3 ---
--- ref 

## Board: ref vs FPGA + batch accuracy

In [3]:
rc = vit.run_vit_smoke(bit_path=BIT, seed=0, n=16)
assert rc == 0, "ViT smoke failed"
print("board vit return", rc)

loaded MNIST sample from /home/xilinx/jupyter_notebooks/mnist_sample.npz (n=16)
=== MNIST tiny-ViT smoke ===
IMG=16 PATCH=4 T=8 D=8 classes=10
scales ACT/W/P=64.0/64.0/127.0
weights=/home/xilinx/jupyter_notebooks/vit_mnist_weights.npz


Using AXI DMA transport (/home/xilinx/jupyter_notebooks/npukit.bit)
ID=0x4E50554B version=0x00000300 features=0x00000003

--- image[0] label=6 ---
--- ref ---
ref pred=6 logits_q12[:4]=[12472, -7875, 12638, -3196]
--- FPGA ---
hw  pred=6 logits_q12[:4]=[12472, -7875, 12638, -3196]
tokens: PASS  max|err|=0  tol=512
block.y_out: PASS  max|err|=127  tol=1024
logits: PASS  max|err|=0  tol=1024

--- image[1] label=8 ---
--- ref ---
ref pred=8 logits_q12[:4]=[8307, -5567, 5386, 1893]
--- FPGA ---
hw  pred=8 logits_q12[:4]=[8282, -5532, 5460, 1870]
tokens: PASS  max|err|=0  tol=512
block.y_out: PASS  max|err|=168  tol=1024
logits: PASS  max|err|=74  tol=1024

--- image[2] label=3 ---
--- ref ---
ref pred=0 logits_q12[:4]=[10888, -13018, 7459, 6739]
--- FPGA ---
hw  pred=0 logits_q12[:4]=[10905, -13019, 7490, 6773]
tokens: PASS  max|err|=0  tol=512
block.y_out: PASS  max|err|=152  tol=1024
logits: PASS  max|err|=55  tol=1024

--- image[3] label=0 ---
--- ref ---
ref pred=0 logits_q12[:4]=[1721